In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import catboost as cb
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
import warnings, gc
warnings.filterwarnings('ignore')

SEED    = 42
N_FOLDS = 10
TARGET  = 'Irrigation_Need'
target_map  = {'Low': 0, 'Medium': 1, 'High': 2}
reverse_map = {0: 'Low', 1: 'Medium', 2: 'High'}

In [2]:
COMP = '/kaggle/input/playground-series-s6e4/'
#ORIG = '/kaggle/input/irrigation-water-requirement-prediction-dataset/'

train = pd.read_csv('/kaggle/input/playground-series-s6e4/train.csv')
test  = pd.read_csv('/kaggle/input/playground-series-s6e4/test.csv')
orig  = pd.read_csv('/kaggle/input/irrigation-water-requirement-prediction-dataset/irrigation_prediction.csv')

train[TARGET] = train[TARGET].map(target_map)
orig[TARGET]  = orig[TARGET].map(target_map)

CATS = ['Soil_Type','Crop_Type','Crop_Growth_Stage','Season',
        'Irrigation_Type','Water_Source','Mulching_Used','Region']
NUMS = ['Soil_pH','Soil_Moisture','Organic_Carbon','Electrical_Conductivity',
        'Temperature_C','Humidity','Rainfall_mm','Sunlight_Hours',
        'Wind_Speed_kmh','Field_Area_hectare','Previous_Irrigation_mm']

print(f'Train: {train.shape}, Test: {test.shape}, Orig: {orig.shape}')

Train: (630000, 21), Test: (270000, 20), Orig: (10000, 20)


In [3]:
def add_features(df_list):
    for df in df_list:
        eps = 1e-6
        # Domain interactions
        df['water_avail']   = df['Soil_Moisture'] + df['Rainfall_mm'] / 300.0
        df['heat_stress']   = df['Temperature_C'] / (df['Soil_Moisture'] + eps)
        df['rain_cool']     = df['Rainfall_mm'] / (df['Temperature_C'] + eps)
        df['wind_evap']     = df['Wind_Speed_kmh'] * df['Temperature_C'] / 100.0
        df['soil_x_rain']   = df['Soil_Moisture'] * df['Rainfall_mm'] / 1000.0
        df['temp_x_dry']    = df['Temperature_C'] * (100 - df['Soil_Moisture']) / 100.0
        # Threshold flags
        df['soil_lt_25']    = (df['Soil_Moisture'] < 25).astype(int)
        df['temp_gt_30']    = (df['Temperature_C'] > 30).astype(int)
        df['rain_lt_300']   = (df['Rainfall_mm'] < 300).astype(int)
        df['wind_gt_10']    = (df['Wind_Speed_kmh'] > 10).astype(int)
        # Logit features (strong priors from domain knowledge)
        df['logit_low']     = (16.3173 - 11.0237*df['soil_lt_25'] - 5.8559*df['temp_gt_30']
                               - 10.8500*df['rain_lt_300'] - 5.8284*df['wind_gt_10'])
        df['logit_high']    = (-20.9697 + 10.6947*df['soil_lt_25'] + 5.8763*df['temp_gt_30']
                               + 10.6958*df['rain_lt_300'] + 5.7444*df['wind_gt_10'])
        # Digit features for key numerics
        for c in ['Soil_Moisture','Temperature_C','Rainfall_mm','Wind_Speed_kmh']:
            for k in range(-1, 3):
                df[f'{c}_dig{k}'] = (df[c] // (10**k) % 10).astype(int)
        # Categorical combos
        df['crop_stage']    = df['Crop_Type'].astype(str)+'_'+df['Crop_Growth_Stage'].astype(str)
        df['soil_region']   = df['Soil_Type'].astype(str)+'_'+df['Region'].astype(str)
        df['season_irr']    = df['Season'].astype(str)+'_'+df['Irrigation_Type'].astype(str)
        df['mulch_water']   = df['Mulching_Used'].astype(str)+'_'+df['Water_Source'].astype(str)
    return df_list

add_features([train, test, orig])

CAT_COMBOS = ['crop_stage','soil_region','season_irr','mulch_water']
NEW_NUMS    = ['water_avail','heat_stress','rain_cool','wind_evap','soil_x_rain',
               'temp_x_dry','soil_lt_25','temp_gt_30','rain_lt_300','wind_gt_10',
               'logit_low','logit_high']

# Encode all cats + combos
ALL_CATS = CATS + CAT_COMBOS
for c in ALL_CATS:
    le = LabelEncoder()
    combined = pd.concat([train[c], test[c], orig[c]]).astype(str)
    le.fit(combined)
    for df in [train, test, orig]:
        df[c] = le.transform(df[c].astype(str))

# Digit feature names
DIG_FEATS = [c for c in train.columns if '_dig' in c]
ALL_FEATS = NUMS + NEW_NUMS + ALL_CATS + DIG_FEATS
print(f'Total features: {len(ALL_FEATS)}')

Total features: 51


In [4]:
# Leak-free TE from original dataset — applied to train, test AND orig
te_cols = CATS
for c in te_cols:
    for cls in [0, 1, 2]:
        col = f'TE_orig_{c}_cls{cls}'
        te_map = orig.groupby(c)[TARGET].apply(lambda x: (x==cls).mean())
        for df in [train, test, orig]:          # <-- orig added here
            df[col] = df[c].map(te_map).fillna(1/3)
        ALL_FEATS.append(col)

print(f'Features after TE: {len(ALL_FEATS)}')

Features after TE: 75


In [5]:
for c in CATS + CAT_COMBOS:
    freq = pd.concat([train[c], orig[c], test[c]]).value_counts(normalize=True)
    for df in [train, test, orig]:
        df[f'FREQ_{c}'] = df[c].map(freq).fillna(0).astype('float32')
    ALL_FEATS.append(f'FREQ_{c}')

print(f'Features after freq encoding: {len(ALL_FEATS)}')
ALL_FEATS = list(dict.fromkeys(ALL_FEATS))  # deduplicate
print(f'Unique features: {len(ALL_FEATS)}')

Features after freq encoding: 87
Unique features: 87


In [6]:
X     = train[ALL_FEATS].values.astype(np.float32)
y     = train[TARGET].values
X_te  = test[ALL_FEATS].values.astype(np.float32)
X_orig = orig[ALL_FEATS].values.astype(np.float32)
y_orig = orig[TARGET].values

class_weights = compute_class_weight('balanced', classes=np.array([0,1,2]), y=y)
print(f'Class weights: {dict(zip(["Low","Medium","High"], class_weights.round(3)))}')

Class weights: {'Low': np.float64(0.568), 'Medium': np.float64(0.878), 'High': np.float64(9.996)}


In [ ]:
lgb_params = {
    'objective': 'multiclass', 'num_class': 3,
    'metric': 'multi_logloss', 'boosting_type': 'dart',
    'n_estimators': 3000, 'learning_rate': 0.05,
    'num_leaves': 63, 'max_depth': -1,
    'min_child_samples': 20, 'subsample': 0.8,
    'colsample_bytree': 0.8, 'reg_alpha': 0.1,
    'reg_lambda': 1.0, 'class_weight': 'balanced',
    'random_state': SEED, 'n_jobs': -1, 'verbose': -1,
    # LightGBM runs on CPU on Kaggle (no OpenCL GPU build available)
    # XGBoost and CatBoost below will use GPU
    }

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_lgb  = np.zeros((len(X), 3))
test_lgb = np.zeros((len(X_te), 3))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f'LGB Fold {fold+1}/{N_FOLDS}', end=' ')
    Xtr  = np.vstack([X[tr_idx], X_orig])
    ytr  = np.concatenate([y[tr_idx], y_orig])
    sw   = compute_sample_weight('balanced', ytr)

    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(Xtr, ytr, sample_weight=sw,
              eval_set=[(X[val_idx], y[val_idx])],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    oof_lgb[val_idx]  = model.predict_proba(X[val_idx])
    test_lgb         += model.predict_proba(X_te) / N_FOLDS
    score = balanced_accuracy_score(y[val_idx], oof_lgb[val_idx].argmax(1))
    print(f'| BA={score:.5f}')
    del model; gc.collect()

lgb_oof_score = balanced_accuracy_score(y, oof_lgb.argmax(1))
print(f'\nLGB OOF Balanced Accuracy: {lgb_oof_score:.5f}')

LGB Fold 1/10 

1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


In [ ]:
cb_params = dict(
    iterations=3000, learning_rate=0.05, depth=8,
    loss_function='MultiClass', eval_metric='Accuracy',
    class_weights=class_weights.tolist(),
    random_seed=SEED, verbose=0, task_type='GPU',
    early_stopping_rounds=50,
)

oof_cat  = np.zeros((len(X), 3))
test_cat = np.zeros((len(X_te), 3))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f'CAT Fold {fold+1}/{N_FOLDS}', end=' ')
    Xtr = np.vstack([X[tr_idx], X_orig])
    ytr = np.concatenate([y[tr_idx], y_orig])

    model = cb.CatBoostClassifier(**cb_params)
    model.fit(Xtr, ytr,
              eval_set=(X[val_idx], y[val_idx]),
              use_best_model=True)
    oof_cat[val_idx]  = model.predict_proba(X[val_idx])
    test_cat         += model.predict_proba(X_te) / N_FOLDS
    score = balanced_accuracy_score(y[val_idx], oof_cat[val_idx].argmax(1))
    print(f'| BA={score:.5f}')
    del model; gc.collect()

cat_oof_score = balanced_accuracy_score(y, oof_cat.argmax(1))
print(f'\nCAT OOF Balanced Accuracy: {cat_oof_score:.5f}')

In [ ]:
xgb_params = dict(
    n_estimators=3000, learning_rate=0.05,
    max_depth=6, subsample=0.8, colsample_bytree=0.8,
    objective='multi:softprob', num_class=3,
    eval_metric='mlogloss', random_state=SEED,
    device='cuda', enable_categorical=False,
    alpha=1, reg_lambda=2, n_jobs=-1,
)

oof_xgb  = np.zeros((len(X), 3))
test_xgb = np.zeros((len(X_te), 3))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f'XGB Fold {fold+1}/{N_FOLDS}', end=' ')
    Xtr  = np.vstack([X[tr_idx], X_orig])
    ytr  = np.concatenate([y[tr_idx], y_orig])
    sw   = compute_sample_weight('balanced', ytr)

    model = xgb.XGBClassifier(**xgb_params)
    model.fit(Xtr, ytr, sample_weight=sw,
              eval_set=[(X[val_idx], y[val_idx])],
              verbose=False)
    oof_xgb[val_idx]  = model.predict_proba(X[val_idx])
    test_xgb         += model.predict_proba(X_te) / N_FOLDS
    score = balanced_accuracy_score(y[val_idx], oof_xgb[val_idx].argmax(1))
    print(f'| BA={score:.5f}')
    del model; gc.collect()

xgb_oof_score = balanced_accuracy_score(y, oof_xgb.argmax(1))
print(f'\nXGB OOF Balanced Accuracy: {xgb_oof_score:.5f}')

In [ ]:
# ── Simple average ──
avg_oof  = (oof_lgb + oof_cat + oof_xgb) / 3
avg_test = (test_lgb + test_cat + test_xgb) / 3
avg_score = balanced_accuracy_score(y, avg_oof.argmax(1))
print(f'Simple average OOF: {avg_score:.5f}')

# ── LR stacking ──
oof_stack  = np.hstack([oof_lgb, oof_cat, oof_xgb])
test_stack = np.hstack([test_lgb, test_cat, test_xgb])
meta_oof   = np.zeros((len(y), 3))
meta_test  = np.zeros((len(X_te), 3))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    meta = LogisticRegression(class_weight='balanced', max_iter=2000, C=1.0, random_state=SEED)
    meta.fit(oof_stack[tr_idx], y[tr_idx])
    meta_oof[val_idx] = meta.predict_proba(oof_stack[val_idx])
    meta_test        += meta.predict_proba(test_stack) / N_FOLDS

stack_score = balanced_accuracy_score(y, meta_oof.argmax(1))
print(f'LR stacking OOF:    {stack_score:.5f}')

# Pick best
if stack_score >= avg_score:
    final_probs = meta_test
    print('Using stacking')
else:
    final_probs = avg_test
    print('Using simple average')

# Save hard labels
sub = pd.read_csv(COMP + 'sample_submission.csv')
sub[TARGET] = [reverse_map[p] for p in final_probs.argmax(1)]
sub.to_csv('submission_stack.csv', index=False)
print('submission_stack.csv saved')
print(sub[TARGET].value_counts())

# Save soft probabilities (USE THESE FOR ENSEMBLING!)
proba_df = pd.DataFrame(final_probs, columns=['proba_Low','proba_Medium','proba_High'])
proba_df.insert(0, 'id', sub['id'].values)
proba_df.to_csv('soft_proba_stack.csv', index=False)
print('soft_proba_stack.csv saved — use this for soft voting ensemble!')